In [ ]:
!pip install -q -U bitsandbytes accelerate transformers

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("1. Re-loading the base Mistral Model & Tokenizer...")
model_id = "mistralai/Mistral-7B-v0.1"

# Configure 4-bit loading just like during training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token, tokenizer.padding_side = tokenizer.eos_token, "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("2. Re-defining the Custom Auditor Architecture Blueprint...")
# The system needs to see this blueprint again so it knows how to structure the weights
class LegalRiskClassifier(nn.Module):
    def __init__(self, hidden_size=4096, num_classes=3):
        super().__init__()
        self.attention_weights = nn.Sequential(nn.Linear(hidden_size, 256), nn.Tanh(), nn.Linear(256, 1))
        self.classifier = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, hidden_states):
        global_view = hidden_states[:, 0, :]
        attn_probs = torch.softmax(self.attention_weights(hidden_states), dim=1)
        selective_view = torch.sum(hidden_states * attn_probs, dim=1)
        return self.classifier(torch.cat([global_view, selective_view], dim=1))

# Initialize an empty brain instance on the GPU
trained_brain = LegalRiskClassifier().to(model.device, dtype=torch.bfloat16)

print("3. Injecting your saved training weights from Google Drive...")
# Now we physically fill that blueprint with your learned mathematical weights
trained_brain.load_state_dict(torch.load("/content/drive/MyDrive/LegalAI_Brain_Epoch_3.pt"))
trained_brain.eval()
print("✅ Model successfully reconstructed and ready for auditing!")

# --- LIVE TEST LOOP ---
print("\n4. Running Live Evaluation...")
sample_clause = "The contractor shall indemnify and hold harmless the client against all liabilities, damages, and legal claims arising from contract breach."
inputs = tokenizer(f"<RISK> {sample_clause}", return_tensors="pt").to(model.device)

with torch.no_grad():
    mistral_outputs = model(input_ids=inputs["input_ids"], output_hidden_states=True, return_dict=True)
    logits = trained_brain(mistral_outputs.hidden_states[-1])
    predicted_class = torch.argmax(logits, dim=-1).item()

risk_mapping = {0: "Low/No Risk", 1: "Medium (Operational/Financial)", 2: "High (Legal/Liability)"}
print(f"\nTested Clause: \"{sample_clause}\"")
print(f"Auditor Decision: {risk_mapping[predicted_class]}")

1. Re-loading the base Mistral Model & Tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

2. Re-defining the Custom Auditor Architecture Blueprint...
3. Injecting your saved training weights from Google Drive...
✅ Model successfully reconstructed and ready for auditing!

4. Running Live Evaluation...

Tested Clause: "The contractor shall indemnify and hold harmless the client against all liabilities, damages, and legal claims arising from contract breach."
Auditor Decision: High (Legal/Liability)
